## Data ingestion
- Ingest data -> Save at raw folder
    - Define selected column and its new name
    - Cast all columns type to str

In [1]:
import os
os.chdir("../")

In [2]:
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction'

### Import library

In [3]:
# import necessary libraries
from pathlib import Path
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

# import project modules
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.data_ingestion_config import DataIngestionConfig
from src.churn_prediction.pydantic.pipeline_config import PipelineConfig
from src.churn_prediction.utils.common import load_single_config, get_execution_date, get_spark
from src.churn_prediction.utils.loaders import load_data
from src.churn_prediction.utils.writers import write_data

25/11/10 14:16:31 WARN Utils: Your hostname, Supawits-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.4 instead (on interface en0)
25/11/10 14:16:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/10 14:16:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
class DataIngester:
    """
    Ingests data based on ingestion configuration.
    """
    
    def __init__(self, config_path: str, execution_date: str = None) -> None:
        """
        Initialize ingester with config file.
        
        Args:
            config_path (str): Path to config YAML file
        """
        try:
            logger.info("Initializing DataIngester...")

            # Load config
            self.config_path = Path(config_path)
            self.pipeline_config = load_single_config(PipelineConfig, self.config_path)
            self.execution_date = get_execution_date(execution_date) if execution_date else datetime.now().strftime("%Y-%m-%d")

            self.ingestion_path = Path(self.pipeline_config.ingestion.config_path)
            self.ingestion_config = load_single_config(DataIngestionConfig, self.ingestion_path)

            self.columns_config = self.ingestion_config.columns

            self.input_path = self.pipeline_config.ingestion.input.get("file_path").replace("${execution_date}", self.execution_date)
            self.output_path = self.pipeline_config.ingestion.output.get("file_path").replace("${execution_date}", self.execution_date)

        except Exception as e:
            logger.error(f"Error initializing DataIngester: {e}")
            raise e

    def select_columns(self, df: DataFrame) -> DataFrame:
        """
        Select columns based on ingestion config.
        
        Args:
            df (DataFrame): Input dataframe

        Returns:
            DataFrame: Processed dataframe
        """
        target_columns = [target_col.target_column for target_col in self.columns_config.values()]
        not_matched_columns = set(target_columns) - set(df.columns)
        if not_matched_columns:
            raise ValueError(f"Columns {not_matched_columns} not found in source data.")

        return df[target_columns]

    def rename_and_cast_columns(self, df: DataFrame) -> DataFrame:
        """
        Rename and cast columns based on ingestion config.

        Args:
            df (DataFrame): Input dataframe

        Returns:
            DataFrame: Processed dataframe
        """
        for col, target_col in self.columns_config.items():
            df = df.withColumnRenamed(target_col.target_column, col)
            df = df.withColumn(col, df[col].cast('string'))
        return df

    def ingester(self) -> None:
        """
        Ingest data from source, process it, and return the dataframe.

        Returns:
            None
        """
        try:
            logger.info("Starting data ingestion...")

            # 1. Load data
            df = load_data(source=self.input_path, header=True)

            # 2. Select columns
            df = self.select_columns(df)

            # 3. Rename and cast columns
            df = self.rename_and_cast_columns(df)

            # 4. Flag save time
            df = df.withColumn("dl_data_dt", F.lit(self.execution_date).cast('date'))
            df = df.withColumn("dl_load_ts", F.lit(datetime.now()))

            # 5. Save raw data
            write_data(df, file_path=self.output_path)

            logger.info("Data ingestion completed.")
        
        except Exception as e:
            logger.error(f"Error during data ingestion: {e}")
            raise e

In [7]:
if __name__ == "__main__":
    config_path = "src/churn_prediction/config/conf/conf_customer_profile_dim.yaml"
    ingester = DataIngester(config_path=config_path)
    df = ingester.ingester()

[ 2025-11-10 14:18:08 ] | churn_prediction | INFO     | 3532516268.py:__init__:14 | Initializing DataIngester...
[ 2025-11-10 14:18:08 ] | churn_prediction | INFO     | common.py:load_single_config:41 | Successfully loaded schema: src/churn_prediction/config/conf/conf_customer_profile_dim.yaml
[ 2025-11-10 14:18:08 ] | churn_prediction | INFO     | common.py:load_single_config:41 | Successfully loaded schema: src/churn_prediction/config/ingestion/customer_profile_dim.yaml
[ 2025-11-10 14:18:08 ] | churn_prediction | INFO     | 3532516268.py:ingester:73 | Starting data ingestion...
[ 2025-11-10 14:18:08 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/user_coop_anonymized.csv
[ 2025-11-10 14:18:08 ] | churn_prediction | INFO     | writers.py:write_data:305 | Writing data to local: data/raw/customer_profile_dim/2025-11-10/
[ 2025-11-10 14:18:09 ] | churn_prediction | INFO     | 3532516268.py:ingester:91 | Data ingestion completed.
